In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../muestra_terra/Terra_example.xlsx")

In [3]:
import re
from collections import Counter

# Asegúrate de tener cargado tu DataFrame como `df`
requests = df['Request'].dropna().astype(str).tolist()

# Patrones para detectar secciones dentro del texto
section_patterns = [
    r"in the ([A-Z\s]+?) section",
    r"in the ([A-Z\s]+?) page",
    r"on the ([A-Z\s]+?) section",
    r"on the ([A-Z\s]+?) page",
    r"in ([A-Z\s]+?) section",
    r"on ([A-Z\s]+?) page"
]

# Extraer y normalizar
found_sections = []
for text in requests:
    for pattern in section_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        found_sections.extend([m.strip().title() for m in matches])

# Contar ocurrencias
section_counts = Counter(found_sections)
print(section_counts.most_common(10))

[('Who We Are', 5), ('The Who We Are', 4), ('The', 1), ('Keeping The Header Copy The Same On Both', 1)]


In [ ]:
import pandas as pd
import random
from faker import Faker
from transformers import pipeline, set_seed
from datetime import date, datetime

# Inicialización
fake = Faker()
set_seed(42)
generator = pipeline("text-generation", model="gpt2")

# Valores reales de Request Type
request_types = ['Copy Revision', 'Design Issues', 'Requested Change', 'New Item']
priorities = ['Low', 'Medium', 'High', 'Critical']
sections = ['hero section', 'footer', 'about us page', 'contact form', 'pricing table', 'FAQ block']
status = ["Complete", "In Progress", "In Queue"]

# Función que genera un texto coherente con el tipo de request
def generate_request(request_type, section):
    prompt_dict = {
        'Copy Revision': f"Please revise the copy in the {section}:",
        'Cesign Issues': f"Fix the layout issue in the {section}:",
        'Requested Change': f"Update the {section} as requested:",
        'New Item': f"Add a new element to the {section}:"
    }
    prompt = prompt_dict.get(request_type, f"Please update the {section}:")
    output = generator(prompt, max_length=40, num_return_sequences=1, do_sample=True, temperature=0.9)[0]['generated_text']
    return output.strip()

# Tema Clientes
num_clients = 25
client_names = list({fake.company() for _ in range(num_clients * 2)})[:num_clients]

# Asignar tamaños aleatorios: 5 grandes, 10 medianos, 10 pequeños
random.shuffle(client_names)
clients = {}
for i, name in enumerate(client_names):
    if i < 5:
        clients[name] = 'large'
    elif i < 15:
        clients[name] = 'medium'
    else:
        clients[name] = 'small'


size_weights = {'large': 5, 'medium': 3, 'small': 1}
weighted_clients = []
for client, size in clients.items():
    weighted_clients.extend([client] * size_weights[size])

# Cosas del browser
browsers = [
    "Chrome",
    "Firefox",
    "Safari",
    "Edge",
    "Opera",
    "Brave",
    "Internet Explorer",
    "Samsung Internet"
]

weighted_browsers = (
    ["Chrome"] * 50 +
    ["Safari"] * 20 +
    ["Firefox"] * 15 +
    ["Edge"] * 8 +
    ["Brave"] * 3 +
    ["Opera"] * 2 +
    ["Internet Explorer"] * 1 +
    ["Samsung Internet"] * 1
)
# Cosas del Page

def generate_client_page(company_name):
    # Limpiar nombre de empresa: minúsculas, sin espacios ni caracteres especiales
    domain = company_name.lower().replace(" ", "").replace(",", "").replace(".", "")
    return f"https://www.{domain}.com"


# Crear datos sintéticos
data = []
for _ in range(2789):  # Cambia el número según cuántos registros quieras
    request_type = random.choice(request_types)
    section = random.choice(sections)
    
    company = random.choice(weighted_clients)
    page = generate_client_page(company)

    data.append({
        "Status" : random.choice(status),
        'Project Name': company,
        "Input Date": fake.date_between( start_date=date(2020, 1, 1), end_date=datetime.today().date()).strftime("%d/%m/%Y"),
        'Requester': fake.name(),
        'Request Type': request_type,
        "Browser" : random.choice(weighted_browsers),
        'Priority': random.choice(priorities),
        "Page" : page,
        'Estimated Time (tokens)': random.choice([1, 2, 3, 5, 8, 13]),
        'Delayed': 'Yes' if random.random() < 0.3 else 'No',
        'Request': generate_request(request_type, section)
    })



# Convertir a DataFrame
df = pd.DataFrame(data)

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eo

In [11]:
df

,Status,Project Name,Input Date,Requester,Request Type,Priority,Estimated Time (hrs),Delayed,Request
0,Complete,Joseph-Johnson,27/04/2025,Renee Walsh,Requested Change,High,3,Yes,Update the hero section as requested: https://...
1,Complete,Peck-Wilson,09/07/2024,Melissa Thompson,Requested Change,Low,3,No,Update the hero section as requested:\n\n<meta...
2,In Queue,Peterson-Hubbard,05/01/2020,Gregory Adkins,Requested Change,Low,5,Yes,Update the hero section as requested: https://...
3,In Queue,Peck-Wilson,24/05/2025,Joshua Barnett,Requested Change,Medium,13,Yes,Update the FAQ block as requested: https://git...
4,Complete,Oconnor-Garcia,20/05/2023,Jessica Ray,Design Issues,Low,5,Yes,Please update the about us page:\n\nWelcome to...
5,In Progress,York LLC,29/12/2020,Crystal Vasquez,Requested Change,Medium,13,Yes,Update the footer as requested:\n\nhttps://gis...
6,In Queue,"Robinson, Bowers and Maynard",29/04/2024,Angel Sanchez,Copy Revision,Medium,2,No,Please revise the copy in the pricing table:\n...
7,In Queue,Oconnor-Garcia,31/05/2020,Katelyn Vasquez,Requested Change,High,1,Yes,Update the FAQ block as requested: https://doc...
8,In Progress,Patel-Boyer,24/01/2020,Marie Gray,Copy Revision,Low,2,No,Please revise the copy in the about us page:\n...
9,In Queue,Medina PLC,21/08/2023,Tasha Steele,Requested Change,Critical,13,No,Update the footer as requested:\n\nThe name of...
